# This is my attempt to clean but I didnt remove except duplicates left the rest so everyone can look at the flagged but most of the flagged was not parralized correctly aka has NaN values 

In [1]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/feryaljadallah/3-level-parallel-samer/parallel_samer.csv")
df

,Original_Sentence,Sentence_Level1,Sentence_Level2,Sentence_Level3,Novel,Chapter,L4,L3,Line,Source
0,الفصل الثاني,الفصل التاني,NaN,NaN,0,2.0,الفصل الثاني,الفصل الثاني,0,SAMER
1,«وكان صباح..,كان يوم جميل.,كان يوماً جميلاً.,NaN,0,2.0,«وكان صباح..,«وكان صباح..,1,SAMER
2,يوما واحدا»,يوم واحد,يوما واحدا.,يوماً واحداً.,0,2.0,يوما واحدا»,يوما واحدا»,1,SAMER
3,قضى فتانا إبراهيم — وهذا اسمه — ليلة هادئة عمي...,إبراهيم قضى ليلة نوم عميق وهادئة، ما عدا حلم ق...,قضى إبراهيم، وهو اسم الفتى، ليلة هادئة ونال قس...,قضى إبراهيم، هذا هو اسمه، ليلة هادئة عميقة الن...,0,2.0,قضى فتانا إبراهيم — وهذا اسمه — ليلة هادئة عمي...,قضى فتانا إبراهيم — وهذا اسمه — ليلة هادئة عمي...,2,SAMER
4,ينحدر على أحد جانبيه نهر جائش،,ينحدر نهرٌ على جانب هذا المكان.,ينحدر نهرٌ على أحد جانبيه.,ينحدر نهرٌ على أحد جانبيه.,0,2.0,ينحدر على أحد جانبيه نهر هائج،,ينحدر على أحد جانبيه نهر هائج،,2,SAMER
...,...,...,...,...,...,...,...,...,...,...
20598,ليته يدري!,ليته يعرف!,ليته يعرف!,ليتَهُ يَدري.,9,8.2,ليته يدري!,ليته يدري!,27,SAMER
20599,إذن لهدأ وجيب قلبها،,لقد هدأت وجاء قلبها.,لقد هدأت وجدبت قلبها.,لذلك هدأت وارتاحت.,9,8.2,إذن لهدأت رجفة قلبها،,إذن لهدأت رجفة قلبها،,27,SAMER
20600,واطمأنت إلى سعادة اليوم والغد.,سأكون سعيداً اليوم وغداً.,واطمئنت إلى سعادة اليوم والغد.,واطمئنت إلى سعادة اليوم والغد.,9,8.2,واطمأنت إلى سعادة اليوم والغد.,واطمأنت إلى سعادة اليوم والغد.,27,SAMER
20601,حسبها أن يذكرها جانبلاط،,جان بلاط ذكرها.,جانبلاط ذكرها.,حسبها جانبلاط تذكرها.,9,8.2,كفاها أن يذكرها جانبلاط،,كفاها أن يذكرها جانبلاط،,27,SAMER


In [2]:
df_clean = df[["Original_Sentence", "Sentence_Level1", "Sentence_Level2","Sentence_Level3"]]

In [3]:
df_clean = df_clean.rename(columns={
    1:"Original_Sentence",
    "Sentence_Level1": "level_1",
    "Sentence_Level2": "level_2",
    "Sentence_Level3": "level_3"
})

In [4]:
df_clean

,Original_Sentence,level_1,level_2,level_3
0,الفصل الثاني,الفصل التاني,NaN,NaN
1,«وكان صباح..,كان يوم جميل.,كان يوماً جميلاً.,NaN
2,يوما واحدا»,يوم واحد,يوما واحدا.,يوماً واحداً.
3,قضى فتانا إبراهيم — وهذا اسمه — ليلة هادئة عمي...,إبراهيم قضى ليلة نوم عميق وهادئة، ما عدا حلم ق...,قضى إبراهيم، وهو اسم الفتى، ليلة هادئة ونال قس...,قضى إبراهيم، هذا هو اسمه، ليلة هادئة عميقة الن...
4,ينحدر على أحد جانبيه نهر جائش،,ينحدر نهرٌ على جانب هذا المكان.,ينحدر نهرٌ على أحد جانبيه.,ينحدر نهرٌ على أحد جانبيه.
...,...,...,...,...
20598,ليته يدري!,ليته يعرف!,ليته يعرف!,ليتَهُ يَدري.
20599,إذن لهدأ وجيب قلبها،,لقد هدأت وجاء قلبها.,لقد هدأت وجدبت قلبها.,لذلك هدأت وارتاحت.
20600,واطمأنت إلى سعادة اليوم والغد.,سأكون سعيداً اليوم وغداً.,واطمئنت إلى سعادة اليوم والغد.,واطمئنت إلى سعادة اليوم والغد.
20601,حسبها أن يذكرها جانبلاط،,جان بلاط ذكرها.,جانبلاط ذكرها.,حسبها جانبلاط تذكرها.


# Kind of cleaned the data so I can work with it 

In [5]:
# Install dependencies if you haven't already
# !pip install pandas openpyxl sentence-transformers scikit-learn pyarabic transformers torch tqdm

import re
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
from transformers import AutoTokenizer, AutoModelForMaskedLM

warnings.filterwarnings("ignore")

In [6]:
import torch
print(torch.cuda.is_available())

True


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# CONSTANTS & THRESHOLDS
# ─────────────────────────────────────────────────────────────────────────────
LEVELS = ["level_1", "level_2", "level_3"]
ALL_COLS = ["Original_Sentence"] + LEVELS

# Models
SEM_MODEL  = "paraphrase-multilingual-mpnet-base-v2"   
BERT_MODEL = "aubmindlab/bert-base-arabertv2"           

# Evaluation Thresholds
SEM_SIM_THRESHOLD  = 0.70   # Below this → meaning drift warning
PERPLEXITY_CEILING = 500    # Above this → grammar concern

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# TEXT UTILS & DATA LOADING
# ─────────────────────────────────────────────────────────────────────────────
def clean_arabic(text) -> str:
    """Strip diacritics and non-Arabic chars."""
    if not isinstance(text, str):
        return ""
    try:
        import pyarabic.araby as araby
        text = araby.strip_diacritics(text)
    except ImportError:
        text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = re.sub(r"[^\u0600-\u06FF\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

def tokenize(text: str) -> list:
    return clean_arabic(text).split()

def load_data(df_or_path) -> pd.DataFrame:
    """Loads data from a file path or accepts an existing DataFrame."""
    if isinstance(df_or_path, str) or isinstance(df_or_path, Path):
        fp = Path(df_or_path)
        df = pd.read_csv(fp) if fp.suffix.lower() == ".csv" else pd.read_excel(fp)
    else:
        df = df_or_path.copy()
        
    df.columns = [c.strip().lower() for c in df.columns]

    missing = [c for c in ALL_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}\nFound: {list(df.columns)}")

    df = df.dropna(subset=ALL_COLS, how="all").reset_index(drop=True)
    return df

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# COMPLEXITY METRICS
# ─────────────────────────────────────────────────────────────────────────────
def complexity_metrics(text) -> dict:
    words = tokenize(text)
    n = len(words)
    if n == 0:
        return dict(word_count=0, avg_word_len=0.0, unique_ratio=0.0, long_word_ratio=0.0, complexity_score=0.0)

    avg_len       = float(np.mean([len(w) for w in words]))
    unique_ratio  = len(set(words)) / n
    long_ratio    = sum(1 for w in words if len(w) > 6) / n

    score = (0.40 * avg_len / 10.0 + 0.30 * unique_ratio + 0.30 * long_ratio)

    return dict(
        word_count      = n,
        avg_word_len    = round(avg_len, 3),
        unique_ratio    = round(unique_ratio, 3),
        long_word_ratio = round(long_ratio, 3),
        complexity_score= round(score, 4),
    )

def add_complexity_metrics(df: pd.DataFrame) -> pd.DataFrame:
    for col in ALL_COLS:
        metrics_series = df[col].apply(complexity_metrics)
        for key in ["word_count", "avg_word_len", "unique_ratio", "long_word_ratio", "complexity_score"]:
            df[f"{key}__{col}"] = metrics_series.apply(lambda m: m[key])
    return df

def check_simplification_order(df: pd.DataFrame) -> pd.DataFrame:
    scores = {col: df[f"complexity_score__{col}"] for col in ALL_COLS}
    df["order_ok__orig_l1"] = scores["level_1"]  <= scores["Original_Sentence"]
    df["order_ok__l1_l2"]   = scores["level_1"]  <= scores["level_2"]
    df["order_ok__l2_l3"]   = scores["level_2"]  <= scores["level_3"]
    
    df["order_valid"] = (df["order_ok__orig_l1"] & df["order_ok__l1_l2"] & df["order_ok__l2_l3"])
    return df

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# SEMANTIC SIMILARITY (PATCHED FOR MISSING DATA)
# ─────────────────────────────────────────────────────────────────────────────
def compute_semantic_similarity(df: pd.DataFrame) -> pd.DataFrame:
    print(f"Loading semantic model: {SEM_MODEL}...")
    model = SentenceTransformer(SEM_MODEL)
    
    # Fill NaN values with empty strings and force them to be strings
    orig_texts = df["Original_Sentence"].fillna("").astype(str).tolist()
    orig_emb = model.encode(orig_texts, show_progress_bar=True, batch_size=32)

    for lvl in LEVELS:
        # Clean the level columns the same way
        lvl_texts = df[lvl].fillna("").astype(str).tolist()
        lvl_emb = model.encode(lvl_texts, show_progress_bar=True, batch_size=32)
        
        # Calculate cosine similarity
        scores = [float(cos_sim([o], [s])[0][0]) for o, s in zip(orig_emb, lvl_emb)]
        df[f"sem_sim__{lvl}"] = [round(s, 4) for s in scores]

    return df

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# GRAMMAR PROXY (AraBERT MLM)
# ─────────────────────────────────────────────────────────────────────────────
def pseudo_perplexity(text: str, tokenizer, model, max_len: int = 128) -> float:
    cleaned = clean_arabic(text)
    if not cleaned: return float("nan")

    inputs = tokenizer(cleaned, return_tensors="pt", truncation=True, max_length=max_len, padding=False)
    input_ids = inputs["input_ids"][0]
    n_tokens = len(input_ids) - 2   
    
    if n_tokens <= 0: return float("nan")

    total_log_prob = 0.0
    with torch.no_grad():
        for i in range(1, len(input_ids) - 1):   
            masked = input_ids.clone()
            masked[i] = tokenizer.mask_token_id
            logits = model(masked.unsqueeze(0)).logits
            log_probs = torch.log_softmax(logits[0, i], dim=-1)
            total_log_prob += log_probs[input_ids[i]].item()

    return float(np.exp(-total_log_prob / n_tokens))

def add_grammar_scores(df: pd.DataFrame) -> pd.DataFrame:
    print(f"Loading grammar model: {BERT_MODEL}...")
    tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
    model = AutoModelForMaskedLM.from_pretrained(BERT_MODEL)
    model.eval()

    for col in ALL_COLS:
        print(f"Calculating perplexity for {col}...")
        df[f"perplexity__{col}"] = df[col].apply(lambda t: pseudo_perplexity(t, tokenizer, model))
        
    return df

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# CUSTOM EVALUATION FLAGS
# ─────────────────────────────────────────────────────────────────────────────
def assign_evaluation_status(df: pd.DataFrame) -> pd.DataFrame:
    statuses = []
    details = []

    for _, row in df.iterrows():
        issues = []

        # 1. Semantic drift check
        for lvl in LEVELS:
            col = f"sem_sim__{lvl}"
            if col in row and not np.isnan(row[col]) and row[col] < SEM_SIM_THRESHOLD:
                issues.append(f"Semantic Drift ({lvl})")

        # 2. Simplification Order check
        if "order_valid" in row and not row["order_valid"]:
            issues.append("Order Violation")

        # 3. Grammar check
        for col_suffix in ALL_COLS:
            col = f"perplexity__{col_suffix}"
            if col in row and not np.isnan(row[col]) and row[col] > PERPLEXITY_CEILING:
                issues.append(f"Grammar Concern ({col_suffix})")

        # Define status based on the number of issues detected
        if len(issues) == 0:
            status = "Pass"
        elif len(issues) == 1:
            status = "Moderate"
        else:
            status = "Flag" # 2 or more issues

        statuses.append(status)
        details.append(" | ".join(issues) if issues else "None")

    df["evaluation_status"] = statuses
    df["issue_details"] = details
    return df

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# MAIN EXECUTION
# ─────────────────────────────────────────────────────────────────────────────
df_raw = df_clean

# Run pipeline
print("1. Computing complexity metrics...")
df_processed = add_complexity_metrics(df_raw)
df_processed = check_simplification_order(df_processed)

print("2. Computing semantic similarity...")
df_processed = compute_semantic_similarity(df_processed)

# Toggle to False if you want to skip the slower Grammar check
RUN_GRAMMAR_CHECK = False 
if RUN_GRAMMAR_CHECK:
    print("3. Computing grammar scores...")
    df_processed = add_grammar_scores(df_processed)

print("4. Assigning evaluation status...")
final_df = assign_evaluation_status(df_processed)

# Display the requested columns to verify
display(final_df.head())

1. Computing complexity metrics...


2. Computing semantic similarity...
Loading semantic model: paraphrase-multilingual-mpnet-base-v2...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Batches:   0%|          | 0/644 [00:00<?, ?it/s]

Batches:   0%|          | 0/644 [00:00<?, ?it/s]

Batches:   0%|          | 0/644 [00:00<?, ?it/s]

4. Assigning evaluation status...


,Original_Sentence,level_1,level_2,level_3,word_count__Original_Sentence,avg_word_len__Original_Sentence,unique_ratio__Original_Sentence,long_word_ratio__Original_Sentence,complexity_score__Original_Sentence,word_count__level_1,...,complexity_score__level_3,order_ok__orig_l1,order_ok__l1_l2,order_ok__l2_l3,order_valid,sem_sim__level_1,sem_sim__level_2,sem_sim__level_3,evaluation_status,issue_details
0,الفصل الثاني,الفصل التاني,NaN,NaN,2,5.500,1.0,0.000,0.5200,2,...,0.0000,True,False,True,False,0.7551,0.3324,0.3324,Flag,Semantic Drift (level_2) | Semantic Drift (lev...
1,«وكان صباح..,كان يوم جميل.,كان يوماً جميلاً.,NaN,2,4.000,1.0,0.000,0.4600,3,...,0.0000,True,True,False,False,0.5359,0.5311,0.2447,Flag,Semantic Drift (level_1) | Semantic Drift (lev...
2,يوما واحدا»,يوم واحد,يوما واحدا.,يوماً واحداً.,2,4.500,1.0,0.000,0.4800,2,...,0.4800,True,True,True,True,0.9808,0.9842,0.9730,Pass,None
3,قضى فتانا إبراهيم — وهذا اسمه — ليلة هادئة عمي...,إبراهيم قضى ليلة نوم عميق وهادئة، ما عدا حلم ق...,قضى إبراهيم، وهو اسم الفتى، ليلة هادئة ونال قس...,قضى إبراهيم، هذا هو اسمه، ليلة هادئة عميقة الن...,23,4.130,1.0,0.087,0.4913,21,...,0.4909,True,True,True,True,0.9143,0.9612,0.9498,Pass,None
4,ينحدر على أحد جانبيه نهر جائش،,ينحدر نهرٌ على جانب هذا المكان.,ينحدر نهرٌ على أحد جانبيه.,ينحدر نهرٌ على أحد جانبيه.,6,4.167,1.0,0.000,0.4667,6,...,0.4600,True,True,True,True,0.9307,0.9491,0.9491,Pass,None


In [21]:
final_df_clean = final_df.dropna()

In [22]:
final_df_clean["evaluation_status"].value_counts()

evaluation_status
Moderate    9821
Flag        5705
Pass        4234
Name: count, dtype: int64

In [14]:
final_df["evaluation_status"].value_counts()

evaluation_status
Moderate    9903
Flag        6457
Pass        4243
Name: count, dtype: int64

In [15]:
# Filter the DataFrame to only show rows with the "Flag" status
flagged_df = final_df[final_df['evaluation_status'] == 'Flag']

# Show the first 10 flagged rows
display(flagged_df.head(10))

,Original_Sentence,level_1,level_2,level_3,word_count__Original_Sentence,avg_word_len__Original_Sentence,unique_ratio__Original_Sentence,long_word_ratio__Original_Sentence,complexity_score__Original_Sentence,word_count__level_1,...,complexity_score__level_3,order_ok__orig_l1,order_ok__l1_l2,order_ok__l2_l3,order_valid,sem_sim__level_1,sem_sim__level_2,sem_sim__level_3,evaluation_status,issue_details
0,الفصل الثاني,الفصل التاني,NaN,NaN,2,5.500,1.000,0.000,0.5200,2,...,0.0000,True,False,True,False,0.7551,0.3324,0.3324,Flag,Semantic Drift (level_2) | Semantic Drift (lev...
1,«وكان صباح..,كان يوم جميل.,كان يوماً جميلاً.,NaN,2,4.000,1.000,0.000,0.4600,3,...,0.0000,True,True,False,False,0.5359,0.5311,0.2447,Flag,Semantic Drift (level_1) | Semantic Drift (lev...
7,وقف الجواد الخبيث فجأة،,وقف الحصان الشرير فجأة.,وقف الجواد فجأة.,وقف الجواد فجأة.,4,5.000,1.000,0.000,0.5000,4,...,0.4733,True,False,True,False,0.6306,0.9272,0.9272,Flag,Semantic Drift (level_1) | Order Violation
13,والجو مطلولا لا تخلص معه الأنفاس.,والسماء تمطر، فلا يمكن رؤية شيء من خلال الضباب...,والسماء صافية ولا يوجد أي ضباب.,والسماء ملبدة بالغيوم لا يتخلص منها الهواء.,6,4.500,1.000,0.167,0.5300,10,...,0.5914,True,True,True,True,0.4864,0.2416,0.7191,Flag,Semantic Drift (level_1) | Semantic Drift (lev...
15,وكثيرا ما ثنته عما يقصد إليه،,ثنيه كثيراً عن ما يريد.,وكثيرا ما يثنيه الآخرون عما يريد فعله.,وكثيرا ما حالت بينه وبين هدفه.,6,4.000,1.000,0.000,0.4600,5,...,0.4600,True,True,False,False,0.6778,0.5544,0.7209,Flag,Semantic Drift (level_1) | Semantic Drift (lev...
20,تكسو الحشائش جانبي مجراها،,الحشائش تتسلل حول جوانب طريقها.,تغطي الأعشاب الجانبين.,تغطي الأعشاب الجانبين.,4,5.750,1.000,0.500,0.6800,5,...,0.7533,True,True,True,True,0.5843,0.5604,0.5604,Flag,Semantic Drift (level_1) | Semantic Drift (lev...
24,وكان المنظر من حوله مؤلفا من عناصر إذا اجتمعت،,كان المنظر من حوله ممتعا.,كان المنظر من حوله عبارة عن مجموعة من العناصر ...,وكان المنظر من حوله مزيجاً من العناصر التي إذا...,9,4.222,0.889,0.111,0.4689,5,...,0.4720,True,True,False,False,0.5555,0.9247,0.9476,Flag,Semantic Drift (level_1) | Order Violation
29,ومنعت الذكرى أن تحرك الأسف على فائت،,الذكرى تذكرنا بالأشياء التي فاتتنا.,الذكرى منعتني من الحزن على ما فات.,ومنعت الذكرى أن تُحَرِّكَ الأسف على فائِتٍ.,7,4.286,1.000,0.000,0.4714,5,...,0.4657,False,False,True,False,0.5742,0.8132,0.8574,Flag,Semantic Drift (level_1) | Order Violation
30,أو الرغبة أن تدفع إلى سعي.,أريد أن أبحث عن أشياء جديدة.,أريد أن أسعى وأبذل جهدي لتحقيق رغباتي.,أريد أن تسعى لتحقيق رغباتك.,6,3.333,1.000,0.000,0.4333,6,...,0.4760,False,True,False,False,0.3046,0.6799,0.7412,Flag,Semantic Drift (level_1) | Semantic Drift (lev...
32,ومن خلفه هو أرض بعضها مرعى فيما يعلم،,ومن خلفه أرض خضراء واسعة.,ومن خلفه أرض بعضها مرعى.,ومن خلفه أرض بعضها مرعى.,8,3.750,1.000,0.000,0.4500,5,...,0.4520,False,False,True,False,0.6095,0.8016,0.8016,Flag,Semantic Drift (level_1) | Order Violation


In [16]:
final_df.to_csv("cleaned_dataset2.csv", index=False)

In [17]:
flagged_df.to_csv("flagged.csv", index=False)

# Allam the flagged

In [18]:
import torch
import gc

# Delete the old semantic model and any previous ALLaM attempts if they exist in memory
try:
    del model
    del tokenizer
    del orig_emb
    del lvl_emb
except NameError:
    pass

# Force Python's garbage collector to run
gc.collect()

# Force PyTorch to empty the GPU cache
torch.cuda.empty_cache()

print(f"Memory cleared! GPU is now free: {torch.cuda.memory_allocated() / 1024**2:.2f} MB used.")

Memory cleared! GPU is now free: 9.12 MB used.


In [19]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "humain-ai/ALLaM-7B-Instruct-preview"

print("1. Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("2. Configuring 4-bit compression...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16, 
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("3. Streaming model directly to GPU (Watch your System RAM!)...")
# These parameters are the strictest possible way to save CPU RAM
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map={"": 0},      # Forces direct-to-GPU loading (GPU 0)
    quantization_config=bnb_config,
    low_cpu_mem_usage=True   # Instructs PyTorch to be extremely careful with RAM
)

model.eval()
print("🎉 Model loaded successfully!")

1. Loading tokenizer...


config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

2. Configuring 4-bit compression...
3. Streaming model directly to GPU (Watch your System RAM!)...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

🎉 Model loaded successfully!


In [20]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "humain-ai/ALLaM-7B-Instruct-preview"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16, 
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading model carefully to prevent RAM crash...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config,
    low_cpu_mem_usage=True  # <-- THIS PREVENTS THE KERNEL CRASH
)

model.eval()
print("Model loaded successfully in 4-bit!")

Loading tokenizer...
Loading model carefully to prevent RAM crash...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [23]:
import pandas as pd

flagged_df = pd.read_csv("/kaggle/working/flagged.csv")

In [24]:
import re

def build_fast_prompt(original, l1, l2, l3):
    return f"""أنت مقيم لغوي صارم. اكتشف أي تلاعب في المعنى أو أخطاء لغوية في التبسيط.
النص الأصلي: "{original}"
المستوى 1: "{l1}"
المستوى 2: "{l2}"
المستوى 3: "{l3}"

انحراف المعنى: هل تغير المعنى الأصلي تماماً؟ هل هناك أخطاء واضحة؟
أجب باختصار شديد جداً باستخدام هذا التنسيق بالضبط (سطرين فقط):
التقييم: [اكتب فقط Pass أو Moderate أو Flag]
السبب: [اكتب السبب في جملة واحدة قصيرة جداً، أو اكتب None]"""

def evaluate_row_with_allam(original, l1, l2, l3):
    orig_str, l1_str, l2_str, l3_str = str(original), str(l1), str(l2), str(l3)
    
    # صائد الـ NaN السريع (يوفر وقت الـ GPU)
    if any(val in ["nan", "NaN", "", "None"] for val in [orig_str, l1_str, l2_str, l3_str]):
        return {"evaluation_status": "Flag", "issue_details": "بيانات مفقودة (NaN)"}

    messages = [
        {"role": "system", "content": "أنت مقيم لغوي دقيق. تجيب بوضوح وباختصار شديد."},
        {"role": "user", "content": build_fast_prompt(orig_str, l1_str, l2_str, l3_str)}
    ]
    
    prompt_text = tokenizer.apply_chat_template(
        messages, 
        add_generation_prompt=True, 
        tokenize=False
    )
    
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=100, 
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
        
    input_length = inputs['input_ids'].shape[1]
    response_tokens = output_ids[0][input_length:]
    response_text = tokenizer.decode(response_tokens, skip_special_tokens=True)
    
    # استخراج البيانات من النص العادي (بدون JSON)
    status = "Flag" # التقييم الافتراضي
    details = "تعذر تحديد السبب"
    
    # 1. البحث عن التقييم
    if re.search(r'(?i)التقييم:\s*Pass', response_text):
        status = "Pass"
    elif re.search(r'(?i)التقييم:\s*Moderate', response_text):
        status = "Moderate"
    elif re.search(r'(?i)التقييم:\s*Flag', response_text):
        status = "Flag"
        
    # 2. البحث عن السبب
    details_match = re.search(r'السبب:\s*(.*)', response_text)
    if details_match:
        details = details_match.group(1).strip()
    
    return {"evaluation_status": status, "issue_details": details}

In [25]:
def process_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    statuses = []
    details = []
    
    print(f"Evaluating {len(df)} rows using ALLaM...")
    
    for index, row in df.iterrows():
        result = evaluate_row_with_allam(
            row.get('Original_Sentence', ''), 
            row.get('level_1', ''), 
            row.get('level_2', ''), 
            row.get('level_3', '')
        )
        
        statuses.append(result.get("evaluation_status", "Flag"))
        details.append(result.get("issue_details", "Parse Error"))
        
        if (index + 1) % 5 == 0:
            print(f"Processed {index + 1}/{len(df)} rows...")

    df["evaluation_status"] = statuses
    df["issue_details"] = details
    return df
final_2df = process_dataframe(flagged_df[["Original_Sentence","level_1","level_2","level_3"]].copy())
display(final_2df[["Original_Sentence", "evaluation_status", "issue_details"]].head())
# Example Execution:
# df_raw = pd.read_csv("your_data.csv")
# final_df = process_dataframe(df_raw)
# display(final_df[["original", "evaluation_status", "issue_details"]].head())

Evaluating 6457 rows using ALLaM...
Processed 5/6457 rows...
Processed 10/6457 rows...
Processed 15/6457 rows...
Processed 20/6457 rows...
Processed 25/6457 rows...
Processed 30/6457 rows...
Processed 35/6457 rows...
Processed 40/6457 rows...
Processed 45/6457 rows...
Processed 50/6457 rows...
Processed 55/6457 rows...
Processed 60/6457 rows...
Processed 65/6457 rows...
Processed 70/6457 rows...
Processed 75/6457 rows...
Processed 80/6457 rows...
Processed 85/6457 rows...
Processed 90/6457 rows...
Processed 95/6457 rows...
Processed 100/6457 rows...
Processed 105/6457 rows...
Processed 110/6457 rows...
Processed 115/6457 rows...
Processed 120/6457 rows...
Processed 125/6457 rows...
Processed 130/6457 rows...
Processed 135/6457 rows...
Processed 140/6457 rows...
Processed 145/6457 rows...
Processed 150/6457 rows...
Processed 155/6457 rows...
Processed 160/6457 rows...
Processed 165/6457 rows...
Processed 170/6457 rows...
Processed 175/6457 rows...
Processed 180/6457 rows...
Processed 18

,Original_Sentence,evaluation_status,issue_details
0,الفصل الثاني,Flag,بيانات مفقودة (NaN)
1,«وكان صباح..,Flag,بيانات مفقودة (NaN)
2,وقف الجواد الخبيث فجأة،,Pass,لا توجد أخطاء واضحة أو انحراف في المعنى.
3,والجو مطلولا لا تخلص معه الأنفاس.,Flag,المعنى الأصلي يشير إلى جو مطير يعيق التنفس، بي...
4,وكثيرا ما ثنته عما يقصد إليه،,Flag,"الجملة ""ثنيه كثيراً عن ما يريد"" غير صحيحة لغوياً."


In [26]:
display(final_2df.head(20))


,Original_Sentence,level_1,level_2,level_3,evaluation_status,issue_details
0,الفصل الثاني,الفصل التاني,NaN,NaN,Flag,بيانات مفقودة (NaN)
1,«وكان صباح..,كان يوم جميل.,كان يوماً جميلاً.,NaN,Flag,بيانات مفقودة (NaN)
2,وقف الجواد الخبيث فجأة،,وقف الحصان الشرير فجأة.,وقف الجواد فجأة.,وقف الجواد فجأة.,Pass,لا توجد أخطاء واضحة أو انحراف في المعنى.
3,والجو مطلولا لا تخلص معه الأنفاس.,والسماء تمطر، فلا يمكن رؤية شيء من خلال الضباب...,والسماء صافية ولا يوجد أي ضباب.,والسماء ملبدة بالغيوم لا يتخلص منها الهواء.,Flag,المعنى الأصلي يشير إلى جو مطير يعيق التنفس، بي...
4,وكثيرا ما ثنته عما يقصد إليه،,ثنيه كثيراً عن ما يريد.,وكثيرا ما يثنيه الآخرون عما يريد فعله.,وكثيرا ما حالت بينه وبين هدفه.,Flag,"الجملة ""ثنيه كثيراً عن ما يريد"" غير صحيحة لغوياً."
5,تكسو الحشائش جانبي مجراها،,الحشائش تتسلل حول جوانب طريقها.,تغطي الأعشاب الجانبين.,تغطي الأعشاب الجانبين.,Moderate,المستوى 1 يغير المعنى الأصلي.
6,وكان المنظر من حوله مؤلفا من عناصر إذا اجتمعت،,كان المنظر من حوله ممتعا.,كان المنظر من حوله عبارة عن مجموعة من العناصر ...,وكان المنظر من حوله مزيجاً من العناصر التي إذا...,Moderate,المستوى 3 يحتوي على خطأ نحوي.
7,ومنعت الذكرى أن تحرك الأسف على فائت،,الذكرى تذكرنا بالأشياء التي فاتتنا.,الذكرى منعتني من الحزن على ما فات.,ومنعت الذكرى أن تُحَرِّكَ الأسف على فائِتٍ.,Flag,الترجمة الثانية تحرف المعنى الأصلي.
8,أو الرغبة أن تدفع إلى سعي.,أريد أن أبحث عن أشياء جديدة.,أريد أن أسعى وأبذل جهدي لتحقيق رغباتي.,أريد أن تسعى لتحقيق رغباتك.,Flag,الجملة الأخيرة غير واضحة المعنى.
9,ومن خلفه هو أرض بعضها مرعى فيما يعلم،,ومن خلفه أرض خضراء واسعة.,ومن خلفه أرض بعضها مرعى.,ومن خلفه أرض بعضها مرعى.,Moderate,تغيير في المعنى الأصلي مع وجود أخطاء لغوية في ...


In [27]:
final_2df["evaluation_status"].value_counts()

evaluation_status
Flag        3942
Moderate    1786
Pass         729
Name: count, dtype: int64

In [28]:
final_2df.to_csv("final_2df.csv", index=False)

In [33]:
final_2df

,Original_Sentence,level_1,level_2,level_3,evaluation_status,issue_details
0,الفصل الثاني,الفصل التاني,NaN,NaN,Flag,بيانات مفقودة (NaN)
1,«وكان صباح..,كان يوم جميل.,كان يوماً جميلاً.,NaN,Flag,بيانات مفقودة (NaN)
2,وقف الجواد الخبيث فجأة،,وقف الحصان الشرير فجأة.,وقف الجواد فجأة.,وقف الجواد فجأة.,Pass,لا توجد أخطاء واضحة أو انحراف في المعنى.
3,والجو مطلولا لا تخلص معه الأنفاس.,والسماء تمطر، فلا يمكن رؤية شيء من خلال الضباب...,والسماء صافية ولا يوجد أي ضباب.,والسماء ملبدة بالغيوم لا يتخلص منها الهواء.,Flag,المعنى الأصلي يشير إلى جو مطير يعيق التنفس، بي...
4,وكثيرا ما ثنته عما يقصد إليه،,ثنيه كثيراً عن ما يريد.,وكثيرا ما يثنيه الآخرون عما يريد فعله.,وكثيرا ما حالت بينه وبين هدفه.,Flag,"الجملة ""ثنيه كثيراً عن ما يريد"" غير صحيحة لغوياً."
...,...,...,...,...,...,...
6452,ولكن الأمير جانبلاط يقيم اليوم في الشام نائبا ...,الأمير جانبلاط يعيش في الشام وهو نائب حلب.,يقيم اليوم الأمير جانبلاط في الشام نائبا عن حلب.,يقيم اليوم الأمير جانبلاط في الشام نائبا عن حلب.,Pass,المعنى الأصلي والمستويات 1 و 2 و 3 متطابقة، ول...
6453,لكأنما أراد أخوها قنصوه أن يحول بينها وبين لقياه،,أخاها كان يريد أن يمنعها من رؤيته.,أخاها كان كأنه يريد أن يمنعها من رؤيته.,لكأنه أراد أختها أن يبعدوه عنها لمنعه من لقائها.,Flag,تلاعب غير دقيق في المعنى.
6454,ومن أين له — وهو بعيد بعيد — أن يعرف أنه الساع...,ومن أين له أن يعرف أنه الساعة أن أماني تحب قاي...,ومن أين له أن يعرف أنه الساعة هو الرجل الوحيد ...,ومن أين له أن يعرف أنه الساعي، وهو البعيد، أن ...,Flag,تلاعب غير واضح في المعنى.
6455,وأم ولده السلطان الناصر!,وأمه تُحضر طعام ولده.,وأمّ ولده السلطان الناصر.,وأمَ أَولَدِه السُلطانَ الناصِرَ.,Flag,تلاعب غير واضح في المعنى.


# Now Concatenate

In [34]:
import pandas as pd

# 1. Load your datasets
# Replace these with your actual file names
df_a = pd.read_csv('/kaggle/working/cleaned_dataset2.csv')
df_b = final_2df.copy()

# 2. Clean the join key (remove whitespace and convert to string)
df_a['Original_Sentence'] = df_a['Original_Sentence'].astype(str).str.strip()
df_b['Original_Sentence'] = df_b['Original_Sentence'].astype(str).str.strip()

# 3. Create a subset of Table B with only the columns you want to update
# We only need the key and the columns to be replaced
columns_to_update = ['Original_Sentence', 'evaluation_status', 'issue_details']
df_b_subset = df_b[columns_to_update].copy()

# 4. Perform a left merge
# This keeps all rows from A and adds columns from B
merged_df = df_a.merge(
    df_b_subset, 
    on='Original_Sentence', 
    how='left', 
    suffixes=('', '_new')
)

# 5. Update the values
# Check if new data exists (is not null) and overwrite
mask = merged_df['evaluation_status_new'].notna()

merged_df.loc[mask, 'evaluation_status'] = merged_df.loc[mask, 'evaluation_status_new']
merged_df.loc[mask, 'issue_details'] = merged_df.loc[mask, 'issue_details_new']

# 6. Drop the temporary columns created by the merge
merged_df = merged_df.drop(columns=['evaluation_status_new', 'issue_details_new'])


In [36]:
# To remove duplicates and keep the first occurrence
merged_df.drop_duplicates(subset=['Original_Sentence'], inplace=True)

In [37]:
merged_df

,Original_Sentence,level_1,level_2,level_3,word_count__Original_Sentence,avg_word_len__Original_Sentence,unique_ratio__Original_Sentence,long_word_ratio__Original_Sentence,complexity_score__Original_Sentence,word_count__level_1,...,complexity_score__level_3,order_ok__orig_l1,order_ok__l1_l2,order_ok__l2_l3,order_valid,sem_sim__level_1,sem_sim__level_2,sem_sim__level_3,evaluation_status,issue_details
0,الفصل الثاني,الفصل التاني,NaN,NaN,2,5.500,1.0,0.000,0.5200,2,...,0.0000,True,False,True,False,0.7551,0.3324,0.3324,Flag,بيانات مفقودة (NaN)
9,«وكان صباح..,كان يوم جميل.,كان يوماً جميلاً.,NaN,2,4.000,1.0,0.000,0.4600,3,...,0.0000,True,True,False,False,0.5359,0.5311,0.2447,Flag,بيانات مفقودة (NaN)
10,يوما واحدا»,يوم واحد,يوما واحدا.,يوماً واحداً.,2,4.500,1.0,0.000,0.4800,2,...,0.4800,True,True,True,True,0.9808,0.9842,0.9730,Pass,NaN
11,قضى فتانا إبراهيم — وهذا اسمه — ليلة هادئة عمي...,إبراهيم قضى ليلة نوم عميق وهادئة، ما عدا حلم ق...,قضى إبراهيم، وهو اسم الفتى، ليلة هادئة ونال قس...,قضى إبراهيم، هذا هو اسمه، ليلة هادئة عميقة الن...,23,4.130,1.0,0.087,0.4913,21,...,0.4909,True,True,True,True,0.9143,0.9612,0.9498,Pass,NaN
12,ينحدر على أحد جانبيه نهر جائش،,ينحدر نهرٌ على جانب هذا المكان.,ينحدر نهرٌ على أحد جانبيه.,ينحدر نهرٌ على أحد جانبيه.,6,4.167,1.0,0.000,0.4667,6,...,0.4600,True,True,True,True,0.9307,0.9491,0.9491,Pass,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47254,ليته يدري!,ليته يعرف!,ليته يعرف!,ليته يشعر!,2,4.000,1.0,0.000,0.4600,2,...,0.4600,True,True,True,True,0.9885,0.9885,0.8273,Pass,NaN
47256,إذن لهدأ وجيب قلبها،,لقد هدأت وجاء قلبها.,لقد هدأت وجدبت قلبها.,لذلك هدأت وارتاحت.,4,4.250,1.0,0.000,0.4700,4,...,0.6000,True,True,True,True,0.8325,0.8309,0.6587,Moderate,Semantic Drift (level_3)
47257,واطمأنت إلى سعادة اليوم والغد.,سأكون سعيداً اليوم وغداً.,واطمئنت إلى سعادة اليوم والغد.,واطمئنت إلى سعادة اليوم والغد.,5,5.000,1.0,0.200,0.5600,4,...,0.5600,True,True,True,True,0.7992,0.9865,0.9865,Pass,NaN
47258,حسبها أن يذكرها جانبلاط،,جان بلاط ذكرها.,جانبلاط ذكرها.,حسبها جانبلاط تذكرها.,4,5.250,1.0,0.250,0.5850,3,...,0.6400,True,True,False,False,0.6938,0.8684,0.8856,Flag,تلاعب غير واضح في المعنى.


In [39]:

# 1. Load your datasets
# Replace these with your actual file names
df_a = pd.read_csv('/kaggle/input/datasets/rayaabualjamal/parrallel-bayyin-cleaned/full_merged_data.csv')
df_b = merged_df

# 1. Drop rows where 'evaluation_status' is 'flag'
# (We check if the column exists first just to be safe, in case only one df has it)
if 'evaluation_status' in df_a.columns:
    df_a = df_a[df_a['evaluation_status'] != 'flag']

if 'evaluation_status' in df_b.columns:
    df_b = df_b[df_b['evaluation_status'] != 'flag']

# 2. Concatenate the two dataframes
# ignore_index=True gives the new combined dataframe a fresh, clean index (0, 1, 2...)
df_combined = pd.concat([df_a, df_b], ignore_index=True)

# 3. Drop duplicated rows based on the 'original sentence' column
# keep='first' keeps the first instance it finds (from df_a). 
# If you want to prioritize df_b's version of the sentence, change this to keep='last'
df_final = df_combined.drop_duplicates(subset=['Original_Sentence'], keep='first')

# 7. Save the result
df_final.to_csv('Final.csv', index=False)

print("Merging complete.")

Merging complete.


In [42]:
df_final = df_combined.drop_duplicates(subset=['Original_Sentence'], keep='first')

In [43]:
df_final

,ID,level_1,level_2,level_3,Original_Sentence,word_count__Original_Sentence,avg_word_len__Original_Sentence,unique_ratio__Original_Sentence,long_word_ratio__Original_Sentence,complexity_score__Original_Sentence,...,complexity_score__level_3,order_ok__orig_l1,order_ok__l1_l2,order_ok__l2_l3,order_valid,sem_sim__level_1,sem_sim__level_2,sem_sim__level_3,evaluation_status,issue_details
0,10100300018,"""عدم الانحياز"" يعني أن تكون محايداً ولا تفضل أ...","""عدم الانحياز"" هو مصطلح سياسي يعني أن تكون محا...","""عدم الانحياز"" هو مصطلح سياسي يشير إلى الحياد.","•• ""عدم الانحياز"" اصطلاح سياسي يعنى الحياد..",6,5.333,1.00,0.167,0.5633,...,0.5175,True,True,True,True,0.7668,0.9523,0.9696,Pass,NaN
1,10100300019,الدول غير المنحازة لا تتعامل مع دول الشرق أو ا...,تلتزم الدول غير المنحازة بعدم التحالف مع دول ا...,تلتزم الدول غير المنحازة بالحياد وعدم التحالف ...,حيث تلتزم الدول غير المنحازة بعدم الارتباط بأي...,14,4.429,1.00,0.143,0.5200,...,0.5617,True,True,True,True,0.8315,0.9379,0.9125,Pass,NaN
2,10100300021,الدول غير المنحازة ليسوا متفرجين، هم يتدخلون و...,الدول غير المنحازة ليس لها موقف محايد في القضا...,بمعنى أن الدول غير المنحازة ليس موقفها موقف مت...,بمعنى أن الدول غير المنحازة لا تقف موقف المتفر...,25,4.560,0.92,0.280,0.5424,...,0.5384,False,False,False,False,0.8765,0.8955,0.9449,Moderate,Order Violation
3,10100300022,"وقد بدأ استخدام مصطلح ""عدم الانحياز"" في مؤتمر ...","وقد بدأ استخدام مصطلح ""عدم الانحياز"" في مؤتمر ...","وقد بدأ استعمال اصطلاح ""عدم الانحياز"" في مؤتمر...",وقد بدأ استعمال اصطلاح (عدم الانحياز) في مؤتمر...,17,4.353,1.00,0.118,0.5094,...,0.5071,True,True,False,False,0.9790,0.9831,0.9931,Moderate,Order Violation
4,10100300057,NaN,هيئة سياسية.,هيئة سياسية.,هيئة سياسية,2,5.000,1.00,0.000,0.5000,...,0.5000,True,True,True,True,0.3297,0.9865,0.9865,Moderate,Semantic Drift (level_1)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41731,NaN,ليته يعرف!,ليته يعرف!,ليته يشعر!,ليته يدري!,2,4.000,1.00,0.000,0.4600,...,0.4600,True,True,True,True,0.9885,0.9885,0.8273,Pass,NaN
41732,NaN,لقد هدأت وجاء قلبها.,لقد هدأت وجدبت قلبها.,لذلك هدأت وارتاحت.,إذن لهدأ وجيب قلبها،,4,4.250,1.00,0.000,0.4700,...,0.6000,True,True,True,True,0.8325,0.8309,0.6587,Moderate,Semantic Drift (level_3)
41733,NaN,سأكون سعيداً اليوم وغداً.,واطمئنت إلى سعادة اليوم والغد.,واطمئنت إلى سعادة اليوم والغد.,واطمأنت إلى سعادة اليوم والغد.,5,5.000,1.00,0.200,0.5600,...,0.5600,True,True,True,True,0.7992,0.9865,0.9865,Pass,NaN
41734,NaN,جان بلاط ذكرها.,جانبلاط ذكرها.,حسبها جانبلاط تذكرها.,حسبها أن يذكرها جانبلاط،,4,5.250,1.00,0.250,0.5850,...,0.6400,True,True,False,False,0.6938,0.8684,0.8856,Flag,تلاعب غير واضح في المعنى.


In [50]:
df_final.isnull().sum()

ID                                     19568
level_1                                    0
level_2                                    0
level_3                                    0
Original_Sentence                          0
word_count__Original_Sentence              0
avg_word_len__Original_Sentence            0
unique_ratio__Original_Sentence            0
long_word_ratio__Original_Sentence         0
complexity_score__Original_Sentence        0
word_count__level_1                        0
avg_word_len__level_1                      0
unique_ratio__level_1                      0
long_word_ratio__level_1                   0
complexity_score__level_1                  0
word_count__level_2                        0
avg_word_len__level_2                      0
unique_ratio__level_2                      0
long_word_ratio__level_2                   0
complexity_score__level_2                  0
word_count__level_3                        0
avg_word_len__level_3                      0
unique_rat

In [48]:
# Drop rows where there are NaN values in 'column_name_1' or 'column_name_2'
df_final.dropna(subset=['level_1', 'level_2',"level_3"], inplace=True)

In [53]:
df_final = df_final[df_final['evaluation_status'] != 'Flag']

In [56]:
df_final = df_final[["Original_Sentence", "level_1", "level_2", "level_3", "evaluation_status","issue_details"]]

In [60]:
df_final.head(50)

,Original_Sentence,level_1,level_2,level_3,evaluation_status,issue_details
0,"•• ""عدم الانحياز"" اصطلاح سياسي يعنى الحياد..","""عدم الانحياز"" يعني أن تكون محايداً ولا تفضل أ...","""عدم الانحياز"" هو مصطلح سياسي يعني أن تكون محا...","""عدم الانحياز"" هو مصطلح سياسي يشير إلى الحياد.",Pass,NaN
1,حيث تلتزم الدول غير المنحازة بعدم الارتباط بأي...,الدول غير المنحازة لا تتعامل مع دول الشرق أو ا...,تلتزم الدول غير المنحازة بعدم التحالف مع دول ا...,تلتزم الدول غير المنحازة بالحياد وعدم التحالف ...,Pass,NaN
2,بمعنى أن الدول غير المنحازة لا تقف موقف المتفر...,الدول غير المنحازة ليسوا متفرجين، هم يتدخلون و...,الدول غير المنحازة ليس لها موقف محايد في القضا...,بمعنى أن الدول غير المنحازة ليس موقفها موقف مت...,Moderate,Order Violation
3,وقد بدأ استعمال اصطلاح (عدم الانحياز) في مؤتمر...,"وقد بدأ استخدام مصطلح ""عدم الانحياز"" في مؤتمر ...","وقد بدأ استخدام مصطلح ""عدم الانحياز"" في مؤتمر ...","وقد بدأ استعمال اصطلاح ""عدم الانحياز"" في مؤتمر...",Moderate,Order Violation
5,••السيارات التي يستعملها الدبلوماسيون.. أي رجا...,"السيارات التي يستخدمها الدبلوماسيون تسمى ""هيئة...",تتحمل السيارات التي يستخدمها الدبلوماسيون جملة...,تتحمل السيارات الدبلوماسية، أي تلك التي يستخدم...,Moderate,Order Violation
6,يقصد بالتنمية.. خطة اقتصادية لزيادة الإنتاج.. ...,التنمية هي خطة لتحسين الحياة وزيادة الأشياء ال...,التنمية هي الخطة الاقتصادية التي تهدف إلى زياد...,التنمية هي الخطة الاقتصادية التي تهدف إلى زياد...,Pass,NaN
7,حضرموت – اليمن الديمقراطية,حضرموت - مكان في اليمن.,حضرموت - اليمن,حضرموت - اليمن الفيدرالية.,Pass,NaN
8,العنقاء هي رمز البعث والخلود في الأساطير المصر...,العنقاء رمز الخلود.,العنقاء هي رمز البعث والخلود في الأساطير المصرية.,العنقاء رمز للبعث والخلود في الأساطير المصرية.,Pass,NaN
9,ويقال إنها لما بلغت 500 سنة أحرقت نفسها وبرزت ...,وقيل أنها عندما بلغت 500 سنة أحرقت نفسها وخرج ...,يقال إنها أحرقت نفسها عندما بلغت 500 سنة، وظهر...,ويقال إنها أحرقت نفسها عندما بلغت خمسمائة عام،...,Moderate,Order Violation
10,وقد اعتبرها العرب أحد المستحيلات الثلاثة، العن...,وقد اعتبرها العرب مستحيلة، مثل الطيران مثلًا!,وقد اعتبرها العرب إحدى المستحيلات الثلاث.,وقد اعتبرها العرب إحدى المستحيلات الثلاث.,Moderate,Order Violation


In [64]:
df_final.to_csv("df_final.csv", index=False)

In [63]:
from IPython.display import FileLink
# Replace 'merged_table.csv' with the actual filename you used
FileLink(df_final)

TypeError: stat: path should be string, bytes, os.PathLike or integer, not DataFrame

In [65]:
import numpy as np

# Split the dataframe into 3 parts
split_list = np.array_split(df_final, 4)

# Save each part
split_list[0].to_csv('merged_part_1.csv', index=False)
split_list[1].to_csv('merged_part_2.csv', index=False)
split_list[2].to_csv('merged_part_3.csv', index=False)
split_list[3].to_csv('merged_part_4.csv', index=False)

print("Split completed successfully. Please download these files individually.")

Split completed successfully. Please download these files individually.


In [57]:
print(f"Success! Total rows: {len(df_final)}")

Success! Total rows: 36723


In [58]:
df_final["evaluation_status"].value_counts()

evaluation_status
Moderate    27461
Pass         9262
Name: count, dtype: int64

In [61]:
df_final.fillna(value="None", inplace=True)

In [62]:
total_nan = df_final.isnull().sum().sum()
print(total_nan)

0


# Hoenstly embarresed by how many tries it took me to figure out I neded to split the file to save it 